# Gold Layer — fact_restaurant_rating_daily_snapshot (Google rating per restaurant × day)

Added 2026-11-01. **Periodic snapshot**, grain = one Google place (restaurant) × one Vietnam day, from the day of its first read to today, **dense**: a day without a read carries the last read forward (same as the YouTube daily snapshots, which also keep a row per day).

Source: `silver.stg_place_rating_reads` (every Place Details / manual Text Search read, append-only). Google only gives the current rating, so the history starts on the first read; restaurants are re-read every 30 days (top 20 by `legit_score`: every 7 days; Google-only: every 90 days) because of the 900 Place Details calls per month.

| Column | Meaning | Additivity |
| --- | --- | --- |
| `snapshot_date_key` | day (yyyyMMdd, Vietnam time) → `dim_date` | |
| `restaurant_key` | current `dim_restaurant` version with this `place_id` (−1 if none) | |
| `place_id` | degenerate dimension | |
| `rating`, `rating_count` | Google stars / number of reviews of the latest read on or before that day | semi-additive: last day of a period, never SUM over days (AVG over restaurants is fine for `rating`) |
| `rating_change`, `rating_count_gained` | vs the previous day; non-zero only on a day with a new read, covering the whole gap since the read before | additive over days (sum = change over the period); NULL on the first day |
| `is_new_read` | a read happened that day (false = carried forward) | |
| `days_since_read` | age of the value shown (0 on a read day) | |
| `read_at`, `read_source` | when / how the shown value was read (`place_details` / `text_search`) | |

**Load:** full rebuild (`CREATE OR REPLACE`, atomic) every run: the table is derived only from the append-only reads, small (~300 places × days), and a rebuild picks up new `restaurant_key`s after an SCD 2 change.

Note: Google's rating is the all-time average, so it moves slowly; `rating_count_gained` (new reviews since the last read) is usually the more telling number.

In [ ]:
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

SNAPSHOT_TZ = "Asia/Ho_Chi_Minh"
READS = f"{SILVER}.stg_place_rating_reads"
TARGET = f"{GOLD}.fact_restaurant_rating_daily_snapshot"
print(f"Silver source: {READS} · Gold target: {TARGET}")

In [ ]:
TODAY = f"to_date(from_utc_timestamp(current_timestamp(), '{SNAPSHOT_TZ}'))"
snapshot_sql = ("""
    WITH day_reads AS (
        -- latest read of each Vietnam day (Place Details and a manual search on the same day: the later one wins)
        SELECT place_id, to_date(from_utc_timestamp(read_at, '{TZ}')) AS read_date,
               max_by(named_struct('rating', rating, 'rating_count', rating_count, 'read_at', read_at, 'source', source), read_at) AS r
        FROM {READS}
        GROUP BY place_id, to_date(from_utc_timestamp(read_at, '{TZ}'))
    ),
    days AS (
        SELECT place_id, explode(sequence(MIN(read_date), {TODAY})) AS snapshot_date
        FROM day_reads GROUP BY place_id
    ),
    filled AS (
        SELECT d.place_id, d.snapshot_date, r.r IS NOT NULL AS is_new_read,
               last_value(r.r, true) OVER (PARTITION BY d.place_id ORDER BY d.snapshot_date
                                           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS r
        FROM days d
        LEFT JOIN day_reads r ON r.place_id = d.place_id AND r.read_date = d.snapshot_date
    ),
    restaurant AS (
        -- current version of the restaurant with that place_id (two restaurants can share a place: lowest key wins)
        SELECT place_id, min(restaurant_key) AS restaurant_key
        FROM {GOLD}.dim_restaurant
        WHERE is_current AND place_id IS NOT NULL
        GROUP BY place_id
    )
    SELECT
        CAST(date_format(f.snapshot_date, 'yyyyMMdd') AS INT)              AS snapshot_date_key,
        COALESCE(dr.restaurant_key, -1)                                    AS restaurant_key,
        f.place_id,
        f.r.rating                                                         AS rating,
        f.r.rating_count                                                   AS rating_count,
        ROUND(f.r.rating - LAG(f.r.rating) OVER w, 2)                      AS rating_change,
        f.r.rating_count - LAG(f.r.rating_count) OVER w                    AS rating_count_gained,
        f.is_new_read,
        LAG(f.snapshot_date) OVER w IS NULL                                AS is_first_snapshot,
        datediff(f.snapshot_date, to_date(from_utc_timestamp(f.r.read_at, '{TZ}'))) AS days_since_read,
        f.r.read_at                                                        AS read_at,
        f.r.source                                                         AS read_source,
        current_timestamp()                                                AS etl_date
    FROM filled f
    LEFT JOIN restaurant dr ON dr.place_id = f.place_id
    WINDOW w AS (PARTITION BY f.place_id ORDER BY f.snapshot_date)
""".replace("{READS}", READS).replace("{TZ}", SNAPSHOT_TZ).replace("{TODAY}", TODAY).replace("{GOLD}", GOLD))

if spark.catalog.tableExists(READS):
    spark.sql(f"""
        CREATE OR REPLACE TABLE {TARGET}
        COMMENT 'Periodic snapshot: 1 Google place (restaurant) x 1 Vietnam day, dense (last read carried forward). rating / rating_count semi-additive, rating_change / rating_count_gained additive.'
        AS {snapshot_sql}
    """)
else:
    print(f"{READS} does not exist yet: {TARGET} stays empty")
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {TARGET} (
            snapshot_date_key INT, restaurant_key BIGINT, place_id STRING, rating DOUBLE, rating_count INT,
            rating_change DOUBLE, rating_count_gained INT, is_new_read BOOLEAN, is_first_snapshot BOOLEAN,
            days_since_read INT, read_at TIMESTAMP, read_source STRING, etl_date TIMESTAMP
        ) USING DELTA
    """)

In [ ]:
s = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT place_id, snapshot_date_key) AS grain, COUNT(DISTINCT place_id) AS places,
           COUNT_IF(restaurant_key = -1) AS unknown_restaurant, COUNT_IF(rating IS NULL) AS no_rating,
           MAX(snapshot_date_key) AS last_day, COUNT_IF(rating_change <> 0) AS rating_changes
    FROM {TARGET}
""").collect()[0]
print(f"📊 {TARGET}: {s['n']} rows · {s['places']} places · last day {s['last_day']} · "
      f"{s['rating_changes']} rating changes · {s['unknown_restaurant']} rows without a restaurant (key -1)")
print("📊 Grain check (place_id x day): " + ("✅" if s["n"] == s["grain"] else "❌ duplicates"))
if s["n"] != s["grain"] or s["no_rating"]:
    raise AssertionError(f"{TARGET}: {s['n'] - s['grain']} duplicate rows, {s['no_rating']} rows without a rating")
display(spark.sql(f"""
    SELECT place_id, snapshot_date_key, rating, rating_change, rating_count, rating_count_gained
    FROM {TARGET} WHERE is_new_read AND NOT is_first_snapshot
    ORDER BY abs(rating_change) DESC, rating_count_gained DESC LIMIT 20
"""))
print("\n--- fact_restaurant_rating_daily_snapshot complete ---")